In [13]:
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, label_binarize
from category_encoders import BinaryEncoder
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

In [14]:
df_applications = pd.read_csv('./data/cleaned_data/applications.csv')

In [15]:
df_transactions = pd.read_csv('./data/cleaned_data/transactions.csv')

In [16]:
df_completion = pd.read_csv('./data/cleaned_data/completions.csv')

In [17]:
from category_encoders import BinaryEncoder

def preprocess_for_pca_binary_encoding(df: pd.DataFrame) -> pd.DataFrame:
    """
    Preprocesses the DataFrame for PCA by:
    1. Encoding non-numeric features using binary encoding
    2. Combining with numeric features
    3. Handling missing values by dropping rows with any NaNs
    4. Standardizing the data

    Parameters:
    df (pd.DataFrame): Raw input DataFrame

    Returns:
    pd.DataFrame: Standardized DataFrame ready for PCA
    """
    numeric_df = df.select_dtypes(include='number')
    categorical_df = df.select_dtypes(exclude='number')

    if not categorical_df.empty:
        encoder = BinaryEncoder()
        encoded_df = encoder.fit_transform(categorical_df)
        full_df = pd.concat([numeric_df, encoded_df], axis=1)
    else:
        full_df = numeric_df

    clean_df = full_df.dropna()

    scaler = StandardScaler()
    standardized_data = scaler.fit_transform(clean_df)

    return pd.DataFrame(standardized_data, columns=clean_df.columns, index=clean_df.index)

In [18]:
def plot_pca_feature_importance(df: pd.DataFrame, original_df: pd.DataFrame, n_components: int = 5, top_n: int = 20):
    """
    Visualizes the most important features using PCA loadings and decodes binary-encoded feature names to their original values.

    Parameters:
    df (pd.DataFrame): DataFrame ready for PCA (will filter to numeric only)
    original_df (pd.DataFrame): Original dataframe before encoding
    n_components (int): Number of principal components to consider
    top_n (int): Number of top features to display
    """
    # Ensure numeric columns only for PCA
    df_numeric = df.select_dtypes(include=[np.number])
    if df.shape[1] != df_numeric.shape[1]:
        dropped = set(df.columns) - set(df_numeric.columns)
        print(f"Warning: Dropped non-numeric columns: {dropped}")

    # Detect categorical columns automatically
    categorical_cols = original_df.select_dtypes(include=['object', 'category']).columns.tolist()

    # Build mappings from encoded column to original label values
    encoded_label_map = {}
    for col in categorical_cols:
        encoder = BinaryEncoder(cols=[col])
        encoder.fit(original_df[[col]])
        unique_values = original_df[col].dropna().unique().tolist()
        for i, encoded in enumerate(encoder.get_feature_names_out([col])):
            category_hint = unique_values[i] if i < len(unique_values) else ""
            encoded_label_map[encoded] = f"{encoded} ({category_hint})"

    # Fit PCA and compute importance
    pca = PCA(n_components=n_components).fit(df_numeric)
    importance = np.sum(pca.components_.T ** 2, axis=1)
    top_features = pd.Series(importance, index=df_numeric.columns).nlargest(top_n)

    # Decode feature labels
    decoded_labels = [encoded_label_map.get(col, col) for col in top_features.index]

    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(decoded_labels, top_features.values)
    plt.title(f"Top {top_n} Most Important Features by PCA")
    plt.ylabel("Sum of Squared Loadings")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [19]:
def get_top_features_by_pca(df: pd.DataFrame, top_k: int = 10, n_components: int = 15) -> list:
    """
    Dynamically selects top-k important features (categorical and numeric) using PCA loadings,
    and visualizes their contributions.

    Parameters:
    df (pd.DataFrame): Input DataFrame with mixed types
    top_k (int): Number of top features to keep
    n_components (int): Number of PCA components to consider

    Returns:
    List[str]: Selected original features based on PCA loadings
    """
    # Drop rows with any missing values
    df = df.dropna()

    # Separate categorical and numeric features
    categorical_df = df.select_dtypes(exclude='number')
    numeric_df = df.select_dtypes(include='number')

    # Binary encode categorical
    encoder = BinaryEncoder()
    encoded_df = encoder.fit_transform(categorical_df)
    feature_names = encoder.get_feature_names_out()
    encoded_df.columns = feature_names

    # Map binary columns to original categorical features
    col_to_feature = {col: col.rsplit('_', 1)[0] for col in encoded_df.columns}

    # Combine numeric and encoded categorical features
    full_df = pd.concat([numeric_df, encoded_df], axis=1)

    # Add numeric feature names to mapping directly
    for col in numeric_df.columns:
        col_to_feature[col] = col

    # Standardize all features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(full_df)

    # PCA
    pca = PCA(n_components=n_components)
    pca.fit(X_scaled)
    loadings = pca.components_.T

    # Compute sum of squared loadings by original feature
    loading_df = pd.DataFrame(loadings, index=full_df.columns)
    loading_df['sum_squared'] = loading_df.iloc[:, :n_components].pow(2).sum(axis=1)

    feature_scores = loading_df.groupby(col_to_feature).sum()['sum_squared'].sort_values(ascending=False)
    top_features = feature_scores.head(top_k).index.tolist()

    # Plot top features
    plt.figure(figsize=(10, 6))
    feature_scores.head(top_k).plot(kind='barh')
    plt.gca().invert_yaxis()
    plt.title(f"Top {top_k} Important Features by PCA (Categorical + Numeric)")
    plt.xlabel("Aggregated Sum of Squared PCA Loadings")
    plt.tight_layout()
    plt.show()

    # Print summary
    print("\nFeature Importance Summary:")
    for feature in top_features:
        score = feature_scores[feature]
        print(f"- {feature}: contributes strongly across the first {n_components} PCA components (score: {score:.4f})")

    return top_features
